In [1]:
# 1. Imports and project paths
import os
import sys
from pathlib import Path

import yaml
import pandas as pd
from tqdm import tqdm
from PIL import Image
import pytesseract

# Point ROOT_DIR to the project root (one level above src/, data/, configs/, notebooks/)
ROOT_DIR = Path("..").resolve()
DATA_DIR = ROOT_DIR

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.eval import cer, wer_jiwer   # uses same metrics as your CRNN


In [2]:
# 2. Configure local Tesseract installation (adjust paths to match your PC)
os.environ["TESSDATA_PREFIX"] = r"C:\Program Files (x86)\Tesseract-OCR\tessdata"
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"


In [3]:
# 3. Load validation CSV specified in configs/default.yaml
config = yaml.safe_load(open(ROOT_DIR / "configs" / "default.yaml", "r"))
val_csv = DATA_DIR / config["data"]["val_csv"]

df_val = pd.read_csv(val_csv)
print(df_val.shape, df_val.columns)
# Expect: (1365, 2) and columns ['page_image_path', 'text']


(1365, 2) Index(['page_image_path', 'text'], dtype='str')


In [4]:
# 4. Fix any mis‑encoded dashes in page_image_path (needed for a few rows)
df_val["page_image_path"] = (
    df_val["page_image_path"]
    .str.replace(" â€“ ", " - ", regex=False)
    .str.replace("â€“", "-", regex=False)
)
# This step is not necessary for everyone, but I included it because some lines in my CSV file needed to be renamed

In [5]:
# 5. Simple Tesseract wrapper
def tesseract_ocr(path: str) -> str:
    img = Image.open(path)
    # PSM 6 = assume a single uniform block of text
    text = pytesseract.image_to_string(img, lang="eng", config="--psm 6")
    return text


In [6]:
# 6. Loop over validation set and compute CER/WER
cers, wers = [], []

for _, row in tqdm(df_val.iterrows(), total=len(df_val)):
    rel_path = row["page_image_path"]
    img_path = DATA_DIR / rel_path

    if not img_path.exists():
        print("Missing file:", img_path)
        continue

    gt_text = str(row["text"])
    pred = tesseract_ocr(str(img_path))

    cers.append(cer(gt_text, pred))
    wers.append(wer_jiwer(gt_text, pred))

print("Tesseract baseline CER:", sum(cers) / len(cers))
print("Tesseract baseline WER:", sum(wers) / len(wers))


 97%|██████████████████████████████████████████████████████████████████████████  | 1330/1365 [1:06:25<01:36,  2.76s/it]C:\Users\Ravindu Fernando\AppData\Local\Programs\Python\Python314\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (94080000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
100%|████████████████████████████████████████████████████████████████████████████| 1365/1365 [1:26:42<00:00,  3.81s/it]

Tesseract baseline CER: 359.6833856722407
Tesseract baseline WER: 255.289700015119
